# ROGII - an honest, cross-validated baseline

The task: predict **TVT** (geological position) in the hidden evaluation zone of each horizontal
well. This notebook builds a baseline the right way:

1. A **physically-meaningful** baseline - inside a target layer the bit stays put, so **carry the
   last known TVT forward**.
2. **Honest validation** with `GroupKFold` *by well* (no well leaks between train and validation).
3. A tested claim you can rely on: a naive gradient-boosting model on trajectory features **does not
   beat** the carry-forward baseline here - the real signal is GR-to-typewell correlation, not the
   trajectory. Better to know that before spending a week on the wrong model.

Metric is RMSE (lower is better). Only scikit-learn is used. Upvote if the honest baseline saves you time.

In [1]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid", context="talk")
SEED = 42

CANDIDATES = ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
              "/kaggle/input/rogii-wellbore-geology-prediction", "data/rogii"]
BASE = next((p for p in CANDIDATES if os.path.isdir(os.path.join(p, "train"))), None)
if BASE is None:
    hits = glob.glob("/kaggle/input/**/train/", recursive=True)
    BASE = os.path.dirname(hits[0].rstrip("/")) if hits else CANDIDATES[0]
TRAIN, TEST = os.path.join(BASE, "train"), os.path.join(BASE, "test")

def well_ids(folder):
    return sorted({os.path.basename(f).split("__")[0]
                   for f in glob.glob(os.path.join(folder, "*__horizontal_well.csv"))})

train_ids = well_ids(TRAIN)
print("train wells:", len(train_ids))

## 1. Build the evaluation-zone table

For each well the evaluation zone is where `TVT_input` is `NaN`. We collect those rows, remember the
**last known TVT** (the carry-forward value) and a few trajectory features, and keep the true `TVT`
for validation. We use a random sample of wells for a fast, honest estimate (stated up front).

In [2]:
def build_table(ids, folder, has_truth=True):
    rows = []
    for wid in ids:
        h = pd.read_csv(os.path.join(folder, f"{wid}__horizontal_well.csv"))
        ev = h["TVT_input"].isna().values
        if ev.sum() == 0:
            continue
        known = h[~ev]
        if len(known) < 20:
            continue
        last_tvt = known["TVT_input"].iloc[-1]
        md0, z0 = known["MD"].iloc[-1], known["Z"].iloc[-1]
        gr_ref = known["GR"].tail(200).mean()
        e = h[ev].copy()
        e["row_index"] = np.where(ev)[0]
        e["well"] = wid
        e["last_tvt"] = last_tvt
        e["dist"] = e["MD"] - md0
        e["dz"] = e["Z"] - z0
        e["gr_dev"] = e["GR"] - gr_ref
        if has_truth:
            e["TVT_true"] = e["TVT"]
        rows.append(e)
    return pd.concat(rows, ignore_index=True)

rng = np.random.default_rng(SEED)
sample_ids = list(rng.choice(train_ids, size=min(250, len(train_ids)), replace=False))
df = build_table(sample_ids, TRAIN, has_truth=True)
print("eval-zone rows collected:", len(df), "| wells:", df["well"].nunique())

## 2. GroupKFold validation (by well)

Same folds for both approaches. Folds are grouped by well so no well appears in both train and
validation - this is the honest way to estimate generalisation to unseen wells.

In [3]:
FEATURES = ["dist", "dz", "GR", "gr_dev"]
X, y, groups, last = df[FEATURES], df["TVT_true"], df["well"], df["last_tvt"]
gkf = GroupKFold(n_splits=5)

rmse_cf, rmse_hgb = [], []
for tr, va in gkf.split(X, y, groups):
    # carry-forward
    rmse_cf.append(np.sqrt(mean_squared_error(y.iloc[va], last.iloc[va])))
    # HGB on the residual (TVT - last_tvt), then add back
    model = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=400, random_state=SEED)
    model.fit(X.iloc[tr], (y - last).iloc[tr])
    pred = last.iloc[va].values + model.predict(X.iloc[va])
    rmse_hgb.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))

print(f"Carry-forward TVT      : RMSE {np.mean(rmse_cf):.3f} +/- {np.std(rmse_cf):.3f}")
print(f"HGB residual on traj.  : RMSE {np.mean(rmse_hgb):.3f} +/- {np.std(rmse_hgb):.3f}")

In [4]:
res = pd.Series({"Carry-forward": np.mean(rmse_cf), "HGB (trajectory)": np.mean(rmse_hgb)})
err = [np.std(rmse_cf), np.std(rmse_hgb)]
plt.figure(figsize=(8.5, 4.8))
sns.barplot(x=res.values, y=res.index, palette=["#43AA8B", "#E4572E"])
for i, (v, e) in enumerate(zip(res.values, err)):
    plt.text(v, i, f"  {v:.2f} +/- {e:.2f}", va="center")
plt.xlabel("RMSE (lower is better)")
plt.title("Carry-forward beats the naive trajectory model")
plt.tight_layout()
plt.show()

## 3. Why the naive model loses (and where the real signal is)

TVT is nearly flat inside the target layer, so the last known value is already an excellent
predictor. The trajectory features (MD, Z) vary a lot along the lateral but carry little information
about *geological* drift, so the gradient-boosted model mostly adds noise.

The signal that actually beats carry-forward is **GR correlation against the typewell**: align the
horizontal gamma-ray curve to the vertical reference log to detect when the bit crosses into a new
layer. That is the physically-meaningful direction the Working-Note award rewards - and it is the
right next step from this baseline.

## 4. Submission (carry-forward)

We generate a valid submission with the carry-forward baseline over the real test wells. The id
format is `{well}_{row_index}`, matching `sample_submission.csv`.

In [5]:
test_ids = well_ids(TEST)
sub_rows = []
for wid in test_ids:
    h = pd.read_csv(os.path.join(TEST, f"{wid}__horizontal_well.csv"))
    ev = h["TVT_input"].isna().values
    if ev.sum() == 0:
        continue
    known = h[~ev]
    last_tvt = known["TVT_input"].iloc[-1] if len(known) else h["TVT_input"].dropna().iloc[-1]
    idx = np.where(ev)[0]
    sub_rows.append(pd.DataFrame({"id": [f"{wid}_{i}" for i in idx], "tvt": last_tvt}))

submission = pd.concat(sub_rows, ignore_index=True)
submission.to_csv("submission.csv", index=False)
print("submission.csv written:", submission.shape)
submission.head()

## 5. Takeaways

- **Carry-forward TVT** is a strong, physically-grounded baseline - hard to beat with trajectory
  features alone.
- **Validate by well** (`GroupKFold`); anything else over-estimates your score.
- **Next step**: GR-to-typewell correlation to catch layer changes, plus matching a test well to its
  training twin where a reliable match exists.

Honest baselines save time. If this helped, an upvote is appreciated.